# Step 0 + Step 1: Compute Check & Dataset Inspection

**Project:** Sickle Cell Detection - CNN research/portfolio tool

**Important framing:** this notebook is part of an **experimental research/screening
tool**, not a medical diagnostic device. Nothing produced here should be interpreted
as a diagnosis or a substitute for laboratory testing or a medical professional.

**What this notebook does:**
1. Confirms we actually have a usable GPU here in Colab (Step 0).
2. Downloads the two datasets into clearly separated folders (never mixed).
3. Inspects their structure, labels, image counts, and license/documentation info
   so we can decide - *before* writing any training code - whether they're
   scientifically usable and how they should be split (Step 1).

**What you should do:** Run the cells top to bottom (`Runtime > Run all`, or one
by one with Shift+Enter). Then copy the printed output back to Claude in the
main conversation so it can review it with you before we build the full pipeline.
Nothing in this notebook trains a model or makes irreversible changes - it only
downloads data and prints information about it.


## 1. Confirm the GPU

**Why:** training two CNNs with cross-validation is only realistic with a GPU.
Colab's free tier *usually* gives you one, but not always (it depends on
availability and your usage quota), so we check rather than assume.

If this prints `No GPU found`, go to `Runtime > Change runtime type` and pick a
GPU (e.g. T4), then re-run this cell before continuing.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || echo "No GPU found by nvidia-smi"


## 2. Get the project code and install dependencies

**Why:** we keep one `requirements.txt` in the GitHub repo so the exact same
package versions are used here as were planned on the development machine -
this avoids "it worked on my machine" version-mismatch bugs later.


In [ ]:
# Clone the project repo (this branch) so requirements.txt and later scripts are available
import os
REPO_URL = "https://github.com/aural0i/Sickle-cell-detection"
BRANCH = "claude/sickle-cell-cnn-research-g6fipc"
if not os.path.isdir("/content/Sickle-cell-detection"):
    !git clone --branch {BRANCH} {REPO_URL} /content/Sickle-cell-detection
%cd /content/Sickle-cell-detection


In [ ]:
!pip install -q -r requirements.txt
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Kaggle API credentials

**Why:** the Kaggle API needs a personal credential tied to your Kaggle account
to download datasets. Kaggle now issues this as a single **API token** (the
kind starting with `KGAT_`) instead of the older downloadable `kaggle.json`
file - the code below uses that new token method.

**How to get it (one-time, per Colab session):**
1. Go to https://www.kaggle.com/settings/api
2. Click "Create New Token" (or similar) to generate a token starting with
   `KGAT_`. Copy it - Kaggle only shows it once.
3. Run the cell below. It will prompt you to **paste the token** using a
   hidden input box (like a password field), so it is never printed on
   screen, saved in this notebook's output, or committed to the repo.

If you generated the older-style `kaggle.json` instead (Kaggle still offers
this under "Legacy API Credentials" on that same page), tell Claude and it
will give you an alternate version of this cell that uses the file-upload
method instead.


In [ ]:
import os
from getpass import getpass

# Paste your KGAT_... token when prompted. It is entered via a hidden field
# and stored only in this Colab runtime's memory (os.environ), not written to
# any file, not printed, and not committed to the repo.
os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (KGAT_...): ")

import kaggle
kaggle.api.authenticate()
print("Kaggle credentials accepted.")


## 4. Download the primary dataset (Kaggle)

**Dataset:** Sickle Cell Disease Dataset (Tushabe et al.)
**Target folder:** `data/train_source/` (kept completely separate from the
external validation data - the task requires we never mix the two).

We also pull the dataset's own metadata (including its stated license) so we
can check the license terms before deciding this is usable in a public
portfolio project.


In [ ]:
import kaggle

os.makedirs("data/train_source", exist_ok=True)
kaggle.api.authenticate()

# Pull dataset metadata (includes license info) without downloading files yet
kaggle.api.dataset_metadata("florencetushabe/sickle-cell-disease-dataset", path=".")
with open("dataset-metadata.json") as f:
    print(f.read())


In [ ]:
# Now download and unzip the actual dataset files
kaggle.api.dataset_download_files(
    "florencetushabe/sickle-cell-disease-dataset",
    path="data/train_source",
    unzip=True,
)
print("Download complete.")


## 5. Download the external validation dataset (Zenodo)

**Dataset:** erythrocytesIDB - https://zenodo.org/records/18299474
**Target folder:** `data/external_val/`

We use Zenodo's own API to list the exact files attached to this record
(including their license) rather than guessing a filename, since Claude
could not view this page directly (network policy in the dev session blocks
Zenodo) and has not seen this record's contents yet.


In [ ]:
import requests

RECORD_ID = "18299474"
r = requests.get(f"https://zenodo.org/api/records/{RECORD_ID}")
r.raise_for_status()
record = r.json()

print("Title:", record["metadata"].get("title"))
print("License:", record["metadata"].get("license"))
print("Access right:", record["metadata"].get("access_right"))
print("Creators:", record["metadata"].get("creators"))
print()
print("Description:")
print(record["metadata"].get("description"))
print()
print("Files in this record:")
for f in record["files"]:
    print(f"  {f['key']}  ({f['size']/1e6:.1f} MB)")


In [ ]:
import os

os.makedirs("data/external_val", exist_ok=True)

for f in record["files"]:
    url = f["links"]["self"]
    out_path = os.path.join("data/external_val", f["key"])
    print(f"Downloading {f['key']} ...")
    resp = requests.get(url, stream=True)
    resp.raise_for_status()
    with open(out_path, "wb") as out:
        for chunk in resp.iter_content(chunk_size=8192):
            out.write(chunk)

print("Download complete. Files saved to data/external_val/")


In [ ]:
# If any downloaded file is a zip/archive, extract it in place
import zipfile, tarfile

for fname in os.listdir("data/external_val"):
    fpath = os.path.join("data/external_val", fname)
    if fname.lower().endswith(".zip"):
        print("Extracting", fname)
        with zipfile.ZipFile(fpath) as z:
            z.extractall("data/external_val")
    elif fname.lower().endswith((".tar", ".tar.gz", ".tgz")):
        print("Extracting", fname)
        with tarfile.open(fpath) as t:
            t.extractall("data/external_val")

print("Done. Current contents of data/external_val:")
print(os.listdir("data/external_val"))


## 6. Inspect folder structure, classes, and image counts

**Why:** before we design a train/val/test split or a leakage-prevention
strategy, we need to actually see how these datasets are organized - are
images grouped by patient/slide/sample? What are the class folder names?
Are there README/LICENSE files bundled inside the download itself?

This cell just *looks*, it doesn't change anything.


In [ ]:
def describe_tree(root, max_depth=3, max_files_per_dir=5):
    root = os.path.abspath(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath) or dirpath}/  ({len(filenames)} files, {len(dirnames)} subfolders)")
        for fn in sorted(filenames)[:max_files_per_dir]:
            print(f"{indent}  - {fn}")
        if len(filenames) > max_files_per_dir:
            print(f"{indent}  ... ({len(filenames) - max_files_per_dir} more files)")

print("=" * 60)
print("PRIMARY DATASET: data/train_source/")
print("=" * 60)
describe_tree("data/train_source")


In [ ]:
print("=" * 60)
print("EXTERNAL DATASET: data/external_val/")
print("=" * 60)
describe_tree("data/external_val")


In [ ]:
# Look for any bundled documentation/license files inside the downloads themselves
import glob

print("Documentation/license-like files found inside data/train_source/:")
for pattern in ["*README*", "*readme*", "*LICENSE*", "*license*", "*.txt", "*.md", "*.pdf", "*.csv"]:
    for p in glob.glob(f"data/train_source/**/{pattern}", recursive=True):
        print(" ", p)

print()
print("Documentation/license-like files found inside data/external_val/:")
for pattern in ["*README*", "*readme*", "*LICENSE*", "*license*", "*.txt", "*.md", "*.pdf", "*.csv"]:
    for p in glob.glob(f"data/external_val/**/{pattern}", recursive=True):
        print(" ", p)


In [ ]:
# Print the contents of any small text/markdown documentation files found, so we can read them here
for pattern in ["data/train_source/**/*README*", "data/train_source/**/*readme*",
                 "data/train_source/**/*LICENSE*", "data/train_source/**/*license*",
                 "data/external_val/**/*README*", "data/external_val/**/*readme*",
                 "data/external_val/**/*LICENSE*", "data/external_val/**/*license*"]:
    for p in glob.glob(pattern, recursive=True):
        try:
            size = os.path.getsize(p)
            if size < 20000:  # only print small text files
                print("=" * 60)
                print(p)
                print("=" * 60)
                with open(p, "r", errors="replace") as fh:
                    print(fh.read())
                print()
        except Exception as e:
            print(f"Could not read {p}: {e}")


In [ ]:
# Count images per top-level class folder, image formats, and image dimensions (sampled)
from PIL import Image
from collections import Counter

def summarize_images(root, label=""):
    print(f"--- Image summary for {label or root} ---")
    ext_counter = Counter()
    class_counter = Counter()
    dims = Counter()
    modes = Counter()
    n_examined = 0
    sample_paths = []

    for dirpath, dirnames, filenames in os.walk(root):
        img_files = [f for f in filenames if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))]
        if not img_files:
            continue
        rel_class = os.path.relpath(dirpath, root)
        class_counter[rel_class] += len(img_files)
        for f in img_files:
            ext_counter[os.path.splitext(f)[1].lower()] += 1
            full = os.path.join(dirpath, f)
            if n_examined < 300:  # sample up to 300 images for dimension/mode check (keep this cell fast)
                try:
                    with Image.open(full) as im:
                        dims[im.size] += 1
                        modes[im.mode] += 1
                except Exception as e:
                    print(f"  Could not open {full}: {e}")
                n_examined += 1
            if len(sample_paths) < 5:
                sample_paths.append(full)

    print("Images per folder (treat as candidate classes):")
    for k, v in sorted(class_counter.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v} images")
    print("File extensions found:", dict(ext_counter))
    print(f"Image dimensions (sampled {n_examined} images):", dict(dims.most_common(10)))
    print("Color modes (sampled):", dict(modes))
    print("Example file paths:")
    for p in sample_paths:
        print(" ", p)
    print()

summarize_images("data/train_source", "PRIMARY (train_source)")
summarize_images("data/external_val", "EXTERNAL (external_val)")


## 7. What to do with this output

Copy everything printed above (or share this notebook) back to Claude in the
main conversation. Claude will use it to:

- Confirm whether the primary dataset is genuinely microscopy imagery suitable
  for this task, and show you the class definitions/counts/example structure
- Determine whether the external dataset (erythrocytesIDB) is scientifically
  compatible enough for external validation, or whether it should be used
  differently (or not at all) - it will **not** force an invalid comparison
- Review the license/usage terms captured above and flag any restriction
  (non-commercial-only, attribution requirements, etc.) before this is used in
  a public portfolio
- Check for patient/slide/sample grouping information needed to prevent data
  leakage when we split the data later

**Nothing has been trained or split yet** - this notebook only downloaded and
described the data.
